In [ ]:
from pacman import Directions
from game import Agent
import api

EmptyLocationValue = 0.1
FoodRewardValue = 10
CapsuleRewardValue = 30
GhostPenaltyValue = -1000

class MDPAgent(Agent):
    def __init__(self):
        print("Starting MDPAgent!")
        self.name = "Pacman"
        self.map = None
        self.walls = None
        self.corners = None
        self.last_move = None  # track last move

    def register_initial_state(self, state):
        """Create static map once at start."""
        pacman_pos = api.whereAmI(state)
        print("Pacman starts at:", pacman_pos)
        self.walls = api.walls(state)
        self.corners = api.corners(state)
        self.map = initialise_map(self.corners, self.walls)

    def getAction(self, state):
        if self.map is None:
            self.register_initial_state(state)

        legal_moves = api.legalActions(state)
        if Directions.STOP in legal_moves:
            legal_moves.remove(Directions.STOP)

        self.map = value_iteration(state)
        pacman_pos = api.whereAmI(state)
        ghosts = api.ghosts(state)

        # Choose which get_best_move function to use
        if len(ghosts) == 0:
            best_action = get_best_move_no_ghosts(
                legal_moves,
                self.map,
                pacman_pos[0],
                pacman_pos[1],
                ghosts,
                self.last_move
            )
        else:
            best_action = get_best_move_with_ghosts(
                legal_moves,
                self.map,
                pacman_pos[0],
                pacman_pos[1],
                ghosts
            )

        self.last_move = best_action  # track last move
        return api.makeMove(best_action, legal_moves)

def initialise_map(corners, walls):
    x_values = [x for x, y in corners]
    y_values = [y for x, y in corners]

    width = max(x_values) + 1
    height = max(y_values) + 1

    pacman_map = []
    for y in range(height):  # rows first
        row = []
        for x in range(width):  # columns
            if (x, y) in walls:
                row.append(None)
            else:
                row.append(EmptyLocationValue)
        pacman_map.append(row)
    return pacman_map


def generate_reward_map(corners, food, ghosts, capsules, walls):
    reward_map = initialise_map(corners, walls)

    x_values = [x for x, y in corners]
    y_values = [y for x, y in corners]

    width = max(x_values) + 1
    height = max(y_values) + 1

    for y in range(height):  # rows
        for x in range(width):  # columns
            current_cell = (x, y)  # (x, y) Pacman coordinates
            # Skip walls, already set to None
            if reward_map[y][x] is None:
                continue
            # Assign rewards
            if current_cell in food:
                reward_map[y][x] = FoodRewardValue
            elif current_cell in ghosts:
                reward_map[y][x] = GhostPenaltyValue
            elif current_cell in capsules:
                reward_map[y][x] = CapsuleRewardValue
            else:
                reward_map[y][x] = EmptyLocationValue
    return reward_map

def has_line_of_sight(walls, start, end):
    """
    Return True if there are no walls directly between start and end.
    Works by stepping along the line in small grid steps.
    """
    x0, y0 = start
    x1, y1 = end

    # Compute direction of movement
    dx = x1 - x0
    dy = y1 - y0

    # Figure out how many steps we’ll take
    steps = max(abs(dx), abs(dy))

    # If start == end, it trivially has line of sight
    if steps == 0:
        return True

    # Calculate the small step in x and y we take each time
    x_step = dx / steps
    y_step = dy / steps

    # Move step-by-step from start → end
    x, y = x0, y0
    for i in range(steps):
        x += x_step
        y += y_step
        check_cell = (round(x), round(y))
        if check_cell in walls:
            return False  # line blocked by wall

    return True


def add_ghost_danger(reward_map, ghost_states, corners, walls):

    any_normal_ghost = any(state == 0 for pos, state in ghost_states)
    if not any_normal_ghost:
        return reward_map

    num_ghosts_in_game = len(ghost_states)

    if num_ghosts_in_game == 1:
        danger_radius = 1
    elif num_ghosts_in_game == 2:
        danger_radius = 3
    else:
        danger_radius = 2

    x_values = [x for x, y in corners]
    y_values = [y for x, y in corners]

    width = max(x_values) + 1
    height = max(y_values) + 1

    all_ghost_positions = [pos for pos, state in ghost_states]

    for gx, gy in all_ghost_positions:
        gx = int(round(gx))
        gy = int(round(gy))

        for dx in range(-danger_radius, danger_radius + 1):
            for dy in range(-danger_radius, danger_radius + 1):
                nx, ny = gx + dx, gy + dy

                # Skip out-of-bounds or wall cells
                if not (0 <= nx < width and 0 <= ny < height):
                    continue
                if (nx, ny) in walls:
                    continue

                # Check line of sight before marking danger
                if not has_line_of_sight(walls, (gx, gy), (nx, ny)):
                    continue  # can't see this square → no danger applied

                distance = abs(dx) + abs(dy)
                if distance == 0:
                    reward_map[ny][nx] = GhostPenaltyValue
                else:
                    danger = GhostPenaltyValue / (distance + 1)
                    reward_map[ny][nx] = min(reward_map[ny][nx], danger)

    return reward_map



def bellman_equation(value_map, current_cell, corners, reward_value):
    gamma_value = 0.95

    x_values = [x for x, y in corners]
    y_values = [y for x, y in corners]

    width = max(x_values) + 1
    height = max(y_values) + 1

    x = current_cell[0]
    y = current_cell[1]

    left = right = up = down = -2

    if value_map[y][x] is None:
        return None

    if x - 1 >= 0:  # inside grid
        if value_map[y][x - 1] is not None:
            left = value_map[y][x - 1]

        # RIGHT neighbor
    if x + 1 < width:
        if value_map[y][x + 1] is not None:
            right = value_map[y][x + 1]

        # UP neighbor
    if y + 1 < height:
        if value_map[y + 1][x] is not None:
            up = value_map[y + 1][x]

        # DOWN neighbor
    if y - 1 >= 0:
        if value_map[y - 1][x] is not None:
            down = value_map[y - 1][x]

    highest_value_neighbour = float(max([up, down, left, right]))
    return float(reward_value + gamma_value * highest_value_neighbour)


def value_iteration(state):
    iterations = 20
    epsilon = 0.01
    capsules = api.capsules(state)
    food = api.food(state)
    walls = api.walls(state)
    corners = api.corners(state)
    ghosts = api.ghosts(state)
    ghost_states_with_times = api.ghostStatesWithTimes(state)
    ghost_states = api.ghostStates(state)

    num_ghosts = len(ghosts)
    if num_ghosts == 0:
        iterations = 40
    elif num_ghosts == 1:
        iterations = 15
    elif num_ghosts == 2:
        iterations = 16

    reward_map = generate_reward_map(corners, food, ghosts, capsules, walls)

    value_map = add_ghost_danger(reward_map, ghost_states, corners, walls)

    for iteration in range(iterations):
        new_map = [row.copy() for row in value_map]
        max_change = 0
        width = max(x for x, y in corners) + 1
        height = max(y for x, y in corners) + 1

        for x in range(width):  # across the columns
            for y in range(height):  # down the rows
                old_value = value_map[y][x]
                reward_value = reward_map[y][x]
                new_value = bellman_equation(value_map, (x, y), corners, reward_value)

                if new_value is None or old_value is None:
                    continue

                new_map[y][x] = new_value
                max_change = max(max_change, abs(new_value - old_value))

        value_map = new_map
        if max_change < epsilon:
            break
    return value_map

def distance(a, ghosts):
    if not ghosts:  # no ghosts present
        return 1000  # arbitrarily large distance
    return min(abs(a[0] - g[0]) + abs(a[1] - g[1]) for g in ghosts)


# Get best move based on value_map, tie-breaking by farthest from ghosts
def get_best_move_with_ghosts(legal_moves, value_map, pacman_x, pacman_y, ghosts):
    best_move = None
    best_value = None
    farthest_dist = -1  # for tie-breaking

    # Ensure ghosts is a list of tuples
    if ghosts is None:
        ghosts = []
    elif isinstance(ghosts, tuple):
        ghosts = [ghosts]

    for move in legal_moves:
        new_x, new_y = pacman_x, pacman_y
        if move == Directions.NORTH:
            new_y += 1
        elif move == Directions.SOUTH:
            new_y -= 1
        elif move == Directions.EAST:
            new_x += 1
        elif move == Directions.WEST:
            new_x -= 1

        # Skip invalid cells
        cell_value = value_map[new_y][new_x]
        if cell_value is None:
            continue

        # Distance to nearest ghost
        ghost_dist = distance((new_x, new_y), ghosts)

        # Choose move with highest value; break ties by farthest from ghosts
        if best_value is None or cell_value > best_value or (cell_value == best_value and ghost_dist > farthest_dist):
            best_value = cell_value
            best_move = move
            farthest_dist = ghost_dist

    return best_move

def get_best_move_no_ghosts(legal_moves, value_map, pacman_x, pacman_y, ghosts, last_move=None):
    """
    Returns the best move based on value_map, with tie-breaking by:
    1. Farthest from ghosts
    2. Avoiding immediate reversal of last move
    """
    best_move = None
    best_value = None
    farthest_dist = -1  # for tie-breaking

    # Map each move to its opposite
    opposite = {
        Directions.NORTH: Directions.SOUTH,
        Directions.SOUTH: Directions.NORTH,
        Directions.EAST: Directions.WEST,
        Directions.WEST: Directions.EAST
    }

    # Ensure ghosts is a list of tuples
    if ghosts is None:
        ghosts = []
    elif isinstance(ghosts, tuple):
        ghosts = [ghosts]

    for move in legal_moves:
        new_x, new_y = pacman_x, pacman_y
        if move == Directions.NORTH:
            new_y += 1
        elif move == Directions.SOUTH:
            new_y -= 1
        elif move == Directions.EAST:
            new_x += 1
        elif move == Directions.WEST:
            new_x -= 1

        # Skip invalid cells
        cell_value = value_map[new_y][new_x]
        if cell_value is None:
            continue

        # Distance to nearest ghost
        ghost_dist = distance((new_x, new_y), ghosts)

        # Apply small penalty for reversing the last move
        adjusted_value = cell_value
        if last_move and move == opposite.get(last_move):
            adjusted_value -= 0.1  # tiny penalty, does NOT override ghost danger

        # Choose move with highest adjusted value; break ties by farthest from ghost
        if best_value is None or adjusted_value > best_value or (adjusted_value == best_value and ghost_dist > farthest_dist):
            best_value = adjusted_value
            best_move = move
            farthest_dist = ghost_dist

    return best_move